# P07 · Cuándo conviene dejar Pandas

> **¿Qué se gana y qué se pierde al leer completo, por bloques, con DuckDB o con Polars?**

stop_times.txt del GTFS vigente: 1 098 717 filas y la misma agregación por parada.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/rendimiento_gtfs/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib 'duckdb>=1.0' 'polars>=1.0,<2'
!python scripts/fetch_data.py gtfs-red-movilidad


## 1 · El problema

«Pandas no escala» y «DuckDB es más rápido» son afirmaciones incompletas. La decisión
depende del volumen, la operación, la memoria disponible y el costo de preparar los
datos. P07 fija una sola agregación y exige que todos los motores devuelvan exactamente
lo mismo antes de comparar tiempos.


## 2 · La solución que se va a aplicar

Se cuentan filas de horario por parada con lectura completa, seis bloques, una consulta
DuckDB y un plan lazy de Polars. Se separa el tiempo de extracción del ZIP de cada
consulta, se mide la memoria del DataFrame o bloque y se hace una unión por clave para
encontrar cualquier diferencia. La conclusión se expresa como criterio, no como ganador
universal.


## Implementaciones comparables

La ruta principal usa Pandas. Polars y R recalculan el indicador central desde el contrato de salida versionado y generan el mismo gráfico.


### Polars + Matplotlib

```python
"""Contraste Polars y visualización Matplotlib del caso rendimiento_gtfs."""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "rendimiento_gtfs" / "resultados" / "benchmark.csv"


def analizar() -> pl.DataFrame:
    datos = pl.read_csv(SOURCE)
    return datos.filter(pl.col("segundos").is_not_null()).sort("segundos").select(["metodo", "segundos"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    x = resultado["metodo"].to_list()
    y = resultado["segundos"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh([str(value) for value in x], y, color="#11253B")
    ax.set_xlabel("Tiempo (segundos; menor es mejor)")
    ax.set_ylabel("Método")
    ax.set_xlim(left=0)
    ax.set_title("Tiempo observado para la misma agregación")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Benchmark P07; tiempos dependientes del equipo · Período: ejecución de referencia",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--grafico", type=Path)
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/rendimiento_gtfs/cuaderno/solucion_polars.py`.


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Contraste R + ggplot2 del caso rendimiento_gtfs.
script_arg <- grep("^--file=", commandArgs(trailingOnly = FALSE), value = TRUE)
script_path <- if (length(script_arg)) sub("^--file=", "", script_arg[1]) else "cuaderno/solucion.R"
root <- normalizePath(file.path(dirname(script_path), "../../.."), mustWork = TRUE)
source_csv <- file.path(root, "cases", "rendimiento_gtfs", "resultados", "benchmark.csv")

datos <- read.csv(source_csv, check.names = FALSE, fileEncoding = "UTF-8")
resultado <- subset(datos, !is.na(segundos), select = c("metodo", "segundos"))[order(subset(datos, !is.na(segundos))$segundos), ]
print(resultado, row.names = FALSE)

args <- commandArgs(trailingOnly = TRUE)
plot_pos <- match("--grafico", args)
if (!is.na(plot_pos)) {
  if (plot_pos == length(args)) stop("--grafico necesita una ruta de salida")
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  destino <- args[plot_pos + 1]
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["metodo"]], y = .data[["segundos"]])
  ) +
    ggplot2::geom_col(fill = "#11253B") +
    ggplot2::coord_flip() +
    ggplot2::labs(
      title = "Tiempo observado para la misma agregación",
      x = "Tiempo (segundos; menor es mejor)", y = "Método",
      caption = "Fuente: Benchmark P07; tiempos dependientes del equipo · Período: ejecución de referencia"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(destino, grafico, width = 9, height = 5, dpi = 150)
  cat(destino, "\n")
}
```

Ejecutar con `Rscript cases/rendimiento_gtfs/cuaderno/solucion.R`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.rendimiento_gtfs import solution as rendimiento

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Fijar una agregación idéntica


Todos cuentan filas por `stop_id`; cambiar la pregunta invalidaría el benchmark.

> ⚠️ Comparar una lectura selectiva con otra que carga ocho columnas mide cosas distintas.


In [ ]:
{'archivo': 'stop_times.txt', 'operacion': 'COUNT(*) GROUP BY stop_id', 'columnas': ['stop_id']}


### Paso 2 · Declarar qué se mide


Tiempo, memoria de datos, bloques y filas se guardan en la misma tabla.

> ⚠️ Un único tiempo total oculta que DuckDB necesita preparar un CSV temporal.


In [ ]:
['segundos','lectura_s','agregacion_s','memoria_datos_mb','bloques','filas_entrada']


### Paso 3 · Ejecutar Pandas completo


Es la referencia simple mientras el archivo cabe en memoria.

> ⚠️ El DataFrame usa más memoria que los 51 MB del CSV porque cada texto es un objeto.


In [ ]:
referencia = rendimiento.cargar_referencia()
referencia['benchmark'].query("metodo == 'Pandas completo'")


### Paso 4 · Ejecutar Pandas por bloques


Cada bloque se agrega y los parciales se suman por clave.

> ⚠️ Una mediana exacta no se puede combinar sumando medianas de bloques.


In [ ]:
referencia['benchmark'].query("metodo == 'Pandas por bloques'")


### Paso 5 · Ejecutar DuckDB y Polars


Ambos consultan el CSV extraído y reportan la extracción aparte.

> ⚠️ DuckDB y Polars lazy no son equivalentes internamente; solo comparten la pregunta y la salida.


In [ ]:
referencia['benchmark'].query("metodo in ['DuckDB sobre CSV', 'Polars lazy sobre CSV']")


### Paso 6 · Exigir igualdad antes de mirar velocidad


Una implementación rápida y distinta es incorrecta.

> ⚠️ Comparar solo el total puede ocultar errores que se compensan entre paradas.


In [ ]:
referencia['validacion']


### Paso 7 · Comparar tiempo y memoria


Ningún método domina las dos dimensiones en todos los volúmenes.

> ⚠️ El color de 'ganador' borra que bloques compra memoria a cambio de tiempo.


In [ ]:
referencia['benchmark']


### Paso 8 · Tomar una decisión proporcional


Se elige el camino más simple que cumple el límite real de recursos.

> ⚠️ Extrapolar linealmente una sola corrida no sustituye medir el siguiente orden de magnitud.


In [ ]:
{'este_feed': 'Pandas completo cabe', 'memoria_restringida': 'bloques', 'muchos_feeds_o_consultas': 'DuckDB/Polars'}


## 4 · El resultado

En este feed Pandas completo cabe y es simple; bloques reduce cerca de 81 % la memoria de datos. DuckDB y Polars ejecutan planes columnares con la preparación informada aparte.

## 5 · Qué queda aprendido

1. Un benchmark empieza por igualdad de resultado.
2. Tiempo y memoria son objetivos distintos.
3. La agregación por bloques exige una operación combinable.
4. Los tiempos se vuelven a medir por equipo.
